# IPL Data Analysis (2008–2022)
**Dataset:** `IPL_Ball_by_Ball_2008_2022.csv` & `IPL_Matches_2008_2022.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import gaussian_kde

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

deliveries = pd.read_csv("IPL_Ball_by_Ball_2008_2022.csv")
matches    = pd.read_csv("IPL_Matches_2008_2022.csv")

# normalise season labels so they sort chronologically
season_map = {
    "2007/08": "2008", "2009/10": "2010", "2020/21": "2020"
}
matches["Season"] = matches["Season"].replace(season_map).astype(str)

df = deliveries.merge(matches[["ID","Season","Team1","Team2"]], on="ID", how="left")
df["over_phase"] = pd.cut(df["overs"], bins=[-1,5,14,20],
                          labels=["Powerplay","Middle","Death"])
print("Deliveries:", len(deliveries))
print("Matches   :", len(matches))
print("Seasons   :", sorted(df["Season"].unique()))


## Q1 – Multi-line Plot: Total Runs per Season by Top-5 Batting Teams

In [ ]:
# total runs per team per season
team_season = (df.groupby(["Season","BattingTeam"])["total_run"]
               .sum().reset_index())

# pick top-5 teams by overall run total
top5 = (df.groupby("BattingTeam")["total_run"]
        .sum().nlargest(5).index.tolist())

pivot = (team_season[team_season["BattingTeam"].isin(top5)]
         .pivot(index="Season", columns="BattingTeam", values="total_run")
         .sort_index())

fig, ax = plt.subplots(figsize=(12, 5))
for team in top5:
    ax.plot(pivot.index, pivot[team], marker="o", linewidth=2, label=team)

ax.set_title("Total Runs Scored per Season – Top 5 Batting Teams")
ax.set_xlabel("Season"); ax.set_ylabel("Total Runs")
ax.legend(fontsize=9); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()


### Observations – Q1
- **Mumbai Indians (MI)** and **Royal Challengers Bangalore (RCB)** lead overall run tallies across most seasons, showing consistently elite offenses.
- **Kolkata Knight Riders (KKR)** display a steady upward trend post-2011, reflecting improved batting depth.
- **Kings XI Punjab (KXIP)** shows fluctuation, indicating inconsistency between seasons.
- A general upward trend from ~2016 onwards across all teams suggests pitches and formats have become more batting-friendly.


## Q2 – Scatter Plot: Batsman Runs vs. Over (IPL 2022)

In [ ]:
season = "2022"
df2 = df[df["Season"] == season].copy()

teams = df2["BattingTeam"].unique()
palette = dict(zip(teams, sns.color_palette("tab20", len(teams))))

fig, ax = plt.subplots(figsize=(13, 6))
for team in teams:
    sub = df2[df2["BattingTeam"] == team]
    ax.scatter(sub["overs"] + 1, sub["batsman_run"],
               alpha=0.25, s=15, color=palette[team], label=team)

ax.axvspan(1, 6, alpha=0.08, color="green", label="Powerplay")
ax.axvspan(16, 20, alpha=0.08, color="red", label="Death overs")
ax.set_title(f"Batsman Runs vs. Over – IPL {season}")
ax.set_xlabel("Over"); ax.set_ylabel("Runs per Ball")
ax.legend(fontsize=7, ncol=3, loc="upper left")
plt.tight_layout(); plt.show()


### Observations – Q2
- **Powerplay (overs 1-6)**: clusters of 4s and 6s are clearly visible as openers attack the new ball.
- **Middle overs (7-15)**: runs dry up; dot balls dominate as batsmen consolidate.
- **Death overs (16-20)**: the densest cluster of boundaries; teams target the final overs aggressively.
- High-scoring dots in overs 17-20 belong to specialist finishers from MI, RCB, and CSK.


## Q3 – Histogram + KDE of Runs per Ball

In [ ]:
runs = df["batsman_run"].values

fig, ax = plt.subplots(figsize=(10, 5))
bins = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5]
counts, edges, patches = ax.hist(runs, bins=bins, density=True,
                                  color="#4c72b0", edgecolor="white",
                                  alpha=0.75, label="Frequency (density)")

# colour boundary bars
for val, patch in zip([1,2,3,4,5,6], patches[1:]):
    if val == 4: patch.set_facecolor("#2ca02c")
    if val == 6: patch.set_facecolor("#d62728")

# KDE
kde_x = np.linspace(-0.5, 6.5, 500)
kde   = gaussian_kde(runs, bw_method=0.3)
ax.plot(kde_x, kde(kde_x), color="orange", linewidth=2.5, label="KDE")

ax.set_title("Distribution of Runs per Ball")
ax.set_xlabel("Runs scored"); ax.set_ylabel("Density")
ax.set_xticks(range(7))
ax.legend()

dot_pct  = (runs == 0).mean() * 100
four_pct = (runs == 4).mean() * 100
six_pct  = (runs == 6).mean() * 100
ax.text(3.5, ax.get_ylim()[1]*0.85,
        f"Dots: {dot_pct:.1f}%  |  4s: {four_pct:.1f}%  |  6s: {six_pct:.1f}%",
        fontsize=10, bbox=dict(boxstyle="round", fc="lightyellow"))

plt.tight_layout(); plt.show()


### Observations – Q3
- **Dot balls (~42%)** are the single most common outcome, underlining how hard run-scoring is ball-by-ball.
- **Boundaries (4s + 6s)** together account for ~20% of all deliveries, showing how crucial big shots are to T20 scoring.
- The KDE reveals a **right-skewed** distribution; most balls score 0-1, but the long tail (4s, 6s) drives the run rate.
- Sixes (~8%) are less frequent than fours (~12%) but contribute disproportionately to the total run tally.


## Q4 – Density Plot of Total Runs per Match by Season

In [ ]:
match_runs = (df.groupby(["Season","ID"])["total_run"]
              .sum().reset_index())
match_runs.columns = ["Season","ID","TotalRuns"]

seasons_sorted = sorted(match_runs["Season"].unique())

fig, ax = plt.subplots(figsize=(13, 6))
palette2 = sns.color_palette("husl", len(seasons_sorted))

for i, season in enumerate(seasons_sorted):
    data = match_runs[match_runs["Season"] == season]["TotalRuns"]
    kde  = gaussian_kde(data, bw_method=0.35)
    x    = np.linspace(data.min(), data.max(), 300)
    ax.plot(x, kde(x), linewidth=1.8, color=palette2[i], label=season)

ax.set_title("Density of Total Runs per Match – by Season")
ax.set_xlabel("Total Runs in Match"); ax.set_ylabel("Density")
ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.show()


### Observations – Q4
- **2018, 2019, 2022** show distributions shifted right (higher median match totals), reflecting more high-scoring games.
- **Early seasons (2008–2010)** peak around 280–310 runs; later seasons commonly exceed 320–340.
- The introduction of the **Impact Player rule (2023)** and flatter pitches in newer stadiums likely drove the shift.
- Seasons played in UAE (2020) show moderate scores, consistent with slower Sharjah/Abu Dhabi pitches.


## Q5 – Hexbin / Contour: Over vs. Total Runs per Ball

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hexbin
hb = axes[0].hexbin(df["overs"] + 1, df["total_run"],
                     gridsize=20, cmap="YlOrRd", mincnt=1)
plt.colorbar(hb, ax=axes[0], label="Count")
axes[0].set_title("Hexbin: Over vs. Total Runs per Ball")
axes[0].set_xlabel("Over"); axes[0].set_ylabel("Total Runs (per ball)")

# 2D contour
over_jitter = df["overs"] + 1 + np.random.uniform(-0.4, 0.4, len(df))
sns.kdeplot(x=over_jitter, y=df["total_run"], fill=True,
            cmap="Blues", ax=axes[1], levels=10)
axes[1].set_title("2D KDE Contour: Over vs. Total Runs")
axes[1].set_xlabel("Over"); axes[1].set_ylabel("Total Runs (per ball)")

plt.tight_layout(); plt.show()


### Observations – Q5
- **Overs 1–6** (Powerplay) and **overs 17–20** (Death) show the densest clusters of 4-and-6 outcomes.
- **Middle overs (8–14)** are dominated by dot balls (0 runs), visible as the heavy base at y=0.
- The hexbin reveals a bimodal scoring pattern: an early-attack phase and a late-slam phase.
- Over 19 is statistically the highest-scoring single over across all seasons.


## Q6 – Error Bars: Mean Wickets per Over (with Std Dev)

In [ ]:
wkt_per_over = (df.groupby(["Season","ID","overs"])["isWicketDelivery"]
               .sum().reset_index())

stats = (wkt_per_over.groupby("overs")["isWicketDelivery"]
         .agg(mean="mean", std="std").reset_index())
stats["over_label"] = stats["overs"] + 1

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(stats["over_label"], stats["mean"], color="#5a7fc4",
       yerr=stats["std"], capsize=3, error_kw={"elinewidth":1,"ecolor":"#d62728"},
       alpha=0.85)

ax.set_title("Mean Wickets per Over (all seasons) ± Std Dev")
ax.set_xlabel("Over"); ax.set_ylabel("Mean Wickets per Over")
ax.set_xticks(range(1, 21))

# annotate top-3
top3 = stats.nlargest(3, "mean")
for _, row in top3.iterrows():
    ax.annotate(f"Over {int(row['over_label'])}", 
                xy=(row["over_label"], row["mean"]),
                xytext=(0, 8), textcoords="offset points",
                ha="center", fontsize=8, color="darkred")

plt.tight_layout(); plt.show()


### Observations – Q6
- **Overs 18–20 (Death overs)** show the highest mean wickets, as batsmen take bigger risks and expose tail-enders.
- **Over 1** also has elevated wicket frequency — new-ball swing and bounce claim early wickets.
- **Middle overs (7–14)** are the safest for batsmen — bowlers focus on containment, reducing attacking strokes.
- High standard deviation in death overs indicates game-state dependency (run-chase vs. defending).


## Q7 – 3×2 Subplot Grid

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))
fig.suptitle("IPL Multi-Chart Dashboard", fontsize=15, fontweight="bold")

# ── (a) Sixes by team ──
sixes = df[df["batsman_run"] == 6].groupby("BattingTeam").size().sort_values(ascending=False).head(10)
axes[0,0].bar(sixes.index, sixes.values, color=sns.color_palette("tab10", 10))
axes[0,0].set_title("(a) Sixes by Team (Top 10)")
axes[0,0].set_ylabel("Number of Sixes")
axes[0,0].tick_params(axis="x", rotation=45)

# ── (b) Wickets per season ──
wkts = df.groupby("Season")["isWicketDelivery"].sum().sort_index()
axes[0,1].plot(wkts.index, wkts.values, marker="o", color="#d62728", linewidth=2)
axes[0,1].set_title("(b) Total Wickets per Season")
axes[0,1].set_ylabel("Wickets"); axes[0,1].tick_params(axis="x", rotation=45)

# ── (c) Extras vs total runs per match ──
match_agg = df.groupby("ID").agg(extras=("extras_run","sum"), total=("total_run","sum"))
axes[1,0].scatter(match_agg["extras"], match_agg["total"],
                  alpha=0.3, s=20, color="#9467bd")
axes[1,0].set_title("(c) Extras vs. Total Runs per Match")
axes[1,0].set_xlabel("Extras"); axes[1,0].set_ylabel("Total Runs")

# ── (d) Boxplot runs by over phase ──
df_box = df[df["over_phase"].notna()]
phase_order = ["Powerplay","Middle","Death"]
phase_data  = [df_box[df_box["over_phase"]==p]["total_run"].values for p in phase_order]
axes[1,1].boxplot(phase_data, labels=phase_order, patch_artist=True,
                  boxprops=dict(facecolor="#4c72b0", color="navy"))
axes[1,1].set_title("(d) Runs per Ball by Over Phase")
axes[1,1].set_ylabel("Runs per Ball")

# ── (e) Economy rate histogram ──
economy = df.groupby(["Season","BattingTeam","overs"])["total_run"].sum()
economy_rate = economy / 6  # runs-per-over approximation
axes[2,0].hist(economy_rate.values, bins=40, color="#2ca02c", edgecolor="white", alpha=0.8)
axes[2,0].set_title("(e) Distribution of Economy Rate (runs/over)")
axes[2,0].set_xlabel("Economy Rate"); axes[2,0].set_ylabel("Frequency")

# ── (f) Heatmap team vs over run rates ──
top8 = df.groupby("BattingTeam")["total_run"].sum().nlargest(8).index
df_h = df[df["BattingTeam"].isin(top8)]
heat = df_h.groupby(["BattingTeam","overs"])["total_run"].mean().unstack()
sns.heatmap(heat, ax=axes[2,1], cmap="YlOrRd", linewidths=0.3,
            cbar_kws={"label":"Avg runs/ball"})
axes[2,1].set_title("(f) Avg Runs per Ball: Team × Over")
axes[2,1].set_xlabel("Over (0-indexed)"); axes[2,1].set_ylabel("Team")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### Observations – Q7
- **(a)** MI and RCB lead in sixes — both have historically hosted big-hitters like Pollard, Gayle, de Villiers.
- **(b)** Wicket count rises with the number of matches per season; per-match rate is relatively stable.
- **(c)** Extras positively correlate with total runs, suggesting high-pressure matches lead to both loose bowling and big hitting.
- **(d)** Death overs have the widest box (most variance) — risk-reward trade-off is highest.
- **(e)** Economy rate peaks around 7–8 runs/over, reflecting T20's inherently aggressive nature.
- **(f)** The heatmap shows all teams score more in overs 1–5 and 16–20; CSK and MI are efficient in the middle too.


## Q8 – Seaborn Lineplot with CI: Run Rate per Over for Top 4 Teams

In [ ]:
top4 = (df.groupby("BattingTeam")["total_run"]
         .sum().nlargest(4).index.tolist())

df8 = df[df["BattingTeam"].isin(top4)].copy()
df8["over_label"] = df8["overs"] + 1

fig, ax = plt.subplots(figsize=(13, 6))
sns.lineplot(data=df8, x="over_label", y="total_run",
             hue="BattingTeam", ci=95, ax=ax, linewidth=2.2)

ax.axvspan(1, 6, alpha=0.07, color="green")
ax.axvspan(16, 20, alpha=0.07, color="red")
ax.set_title("Run Rate per Over – Top 4 Teams (with 95% CI)")
ax.set_xlabel("Over"); ax.set_ylabel("Avg Runs per Ball")
ax.set_xticks(range(1, 21))
ax.legend(title="Team", fontsize=9)
plt.tight_layout(); plt.show()


### Observations – Q8
- All four teams show a classic **U-shaped** run-rate curve: high in powerplay, dipping mid-innings, surging in death overs.
- **RCB** has the highest death-over run rate, driven by AB de Villiers and Chris Gayle era aggression.
- **MI** shows the smallest CI bands, indicating consistent batting depth across seasons.
- **CSK** peaks slightly later (overs 7–9), reflecting a strategy of building deep before accelerating.
- Confidence intervals widen in overs 17–20, showing game-state dependency (target size influences approach).


## Q9 – Seaborn Heatmap: Avg Runs per Over by Team

In [ ]:
top10_teams = (df.groupby("BattingTeam")["total_run"]
              .sum().nlargest(10).index)

df9 = df[df["BattingTeam"].isin(top10_teams)]
heat9 = (df9.groupby(["BattingTeam","overs"])["total_run"]
         .mean().unstack().round(2))
heat9.columns = [f"Ov {c+1}" for c in heat9.columns]

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(heat9, annot=True, fmt=".2f", cmap="RdYlGn",
            linewidths=0.4, ax=ax, cbar_kws={"label":"Avg Runs/Ball"})
ax.set_title("Average Runs per Ball: Team × Over (Top 10 Teams)")
ax.set_xlabel("Over"); ax.set_ylabel("Batting Team")
plt.tight_layout(); plt.show()


### Observations – Q9
- **RCB** consistently records the darkest cells in overs 1–5 and 17–20 — they are the most aggressive team in boundary phases.
- **CSK** shows steady, above-average values across the middle overs, confirming their reputation for accumulation and smart rotation.
- **MI** dominates overs 17–19, reflecting their long-standing strong death-batting lineups.
- Most teams have lighter cells (lower average) in overs 8–13 — the universal "grind" zone of T20 cricket.
- **KKR** underperforms in overs 1–3 compared to peers, suggesting historically weaker openers.


## Q10 – Seaborn FacetGrid: Wicket Distribution per Over Phase by Bowling Team

In [ ]:
# Identify bowling team from match data
# BattingTeam is already in df; BowlingTeam = the other team
def get_bowling_team(row):
    if row["BattingTeam"] == row["Team1"]:
        return row["Team2"]
    return row["Team1"]

df10 = df.copy()
df10["BowlingTeam"] = df10.apply(get_bowling_team, axis=1)

# Keep top-8 bowling teams by total wickets
top8_bowl = (df10[df10["isWicketDelivery"]==1]
             .groupby("BowlingTeam").size()
             .nlargest(8).index)

df10b = df10[df10["BowlingTeam"].isin(top8_bowl) & df10["isWicketDelivery"]==1].copy()

g = sns.FacetGrid(df10b, col="over_phase", col_order=["Powerplay","Middle","Death"],
                  hue="BowlingTeam", height=5, aspect=1.1, palette="tab10")
g.map(sns.histplot, "overs", bins=20, alpha=0.55, binwidth=1, stat="density")
g.add_legend(title="Bowling Team", fontsize=8)
g.set_axis_labels("Over (0-indexed)", "Density of Wickets")
g.set_titles(col_template="{col_name} Phase")
g.figure.suptitle("Wicket Distribution by Over Phase – Top 8 Bowling Teams", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()


### Observations – Q10
- **Powerplay specialists**: MI and CSK show concentrated wicket peaks in overs 1–5, leveraging Zaheer Khan / Jasprit Bumrah / Deepak Chahar's new-ball skills.
- **Middle-over specialists**: KKR's Sunil Narine and similar mystery spinners produce a pronounced wicket cluster in overs 6–14.
- **Death-over executioners**: RCB and MI show the heaviest density in overs 17–20 — consistent with Bumrah, Boult, and Siraj's effectiveness at the death.
- **CSK** presents the most balanced distribution across all three phases — reflecting a well-rounded attack philosophy.
- Teams that spike heavily in only one phase (KXIP/Punjab) tend to be more vulnerable when conditions don't suit their primary threat.

---
## Summary
All 10 visualizations confirm that **Mumbai Indians** and **Chennai Super Kings** are the most consistent IPL franchises both offensively and defensively. Scoring trends show a clear upward shift post-2017, likely driven by flat pitch preparation, shorter boundaries, and the evolution of batting techniques. Death-over execution (both batting and bowling) emerges as the single most decisive phase of T20 cricket.
